# AutoData Agent Colab Runner

This notebook runs the AutoData research pipeline from the plan:

`evaluate -> diagnose -> plan -> generate -> verify -> mix -> fine-tune -> re-evaluate`

Start with `RUN_MODE = "smoke"`. After that succeeds, switch to `prototype` with real MedMCQA and Qwen evaluation/training.

## 1. No Google Drive Required

This notebook saves outputs to the Colab runtime by default, so it does not mount or require Google Drive.

In [ ]:
from pathlib import Path

LOCAL_OUTPUT_DIR = Path('/content/autodata_outputs')
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Outputs will be saved under:', LOCAL_OUTPUT_DIR)

## 2. Locate Or Clone The Repository

If you opened this notebook directly in Colab, it clones the repository into `/content/autodata-agent`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Jiaqi-Ye/AutoData.git'
REPO_DIR = '/content/autodata-agent'

current = Path.cwd()
if (current / 'autodata').exists() and (current / 'configs').exists():
    repo_path = current
else:
    repo_path = Path(REPO_DIR)
    if not (repo_path / 'autodata').exists():
        if not REPO_URL.strip():
            raise RuntimeError(
                'Set REPO_URL to your autodata-agent GitHub repo.'
            )
        if repo_path.exists():
            shutil.rmtree(repo_path)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_path)], check=True)

os.chdir(repo_path)
print('Repository:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:10])

## 3. Install Dependencies

Smoke mode installs only lightweight packages. Prototype/full mode needs the ML stack. On Colab GPU, `bitsandbytes` is supported.

In [ ]:
INSTALL_DEPS = True

LIGHTWEIGHT_DEPS = [
    'pyyaml>=6.0',
    'pytest>=7.0',
    'tqdm>=4.66',
]

REAL_DEPS = [
    'pyyaml>=6.0',
    'pytest>=7.0',
    'tqdm>=4.66',
    'pyarrow>=16.1.0',
    'datasets>=2.18',
    'torch>=2.1',
    'transformers>=4.43',
    'accelerate>=0.31',
    'peft>=0.11',
    'trl>=0.9',
    'bitsandbytes>=0.43',
    'torchao==0.16.0',
]

if INSTALL_DEPS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *LIGHTWEIGHT_DEPS])
print('Dependency step complete.')

## 4. Choose Run Settings

Recommended first run: keep the defaults and run smoke. For the real prototype, use:

- `RUN_MODE = 'prototype'`
- `USE_REAL_MODEL = True`
- `USE_REAL_TRAINING = True`
- keep `USE_MOCK_GENERATION = True` for a cheaper first real-model test, or set it to `False` with `GENERATION_PROVIDER = 'local_hf'` when you are ready to generate with Qwen2.5-7B.

In [ ]:
RUN_MODE = 'smoke'  # 'smoke', 'prototype', or 'full'
RUN_STRATEGY_COMPARISON = False
STRATEGIES = 'uniform,weakness_based,agent_guided'

USE_REAL_MODEL = False
USE_REAL_TRAINING = False
USE_MOCK_GENERATION = True
GENERATION_PROVIDER = 'mock'  # 'mock', 'local_hf', or 'strong_local'

OUTPUT_DIR = '/content/autodata_outputs'

# Set any of these to 0 to keep the YAML default for the selected mode.
EVAL_SAMPLES_PER_DOMAIN = 0
TRAIN_POOL_SAMPLES_PER_DOMAIN = 0
TOTAL_SYNTHETIC_BUDGET = 0
MIN_SAMPLES_PER_DOMAIN = 0
MAX_TRAINING_STEPS = 0

print('Run mode:', RUN_MODE)
print('Strategy comparison:', RUN_STRATEGY_COMPARISON)

## 5. Install Real-Run Dependencies If Needed

This cell only installs the larger ML stack when your settings require it.

In [ ]:
needs_real_stack = USE_REAL_MODEL or USE_REAL_TRAINING or (not USE_MOCK_GENERATION)
real_deps_flag = Path('/content/.autodata_real_deps_ready')

if INSTALL_DEPS and needs_real_stack and not real_deps_flag.exists():
    # Colab can preload an incompatible pyarrow binary before datasets imports it.
    # Reinstall pyarrow/datasets together, then restart the runtime once.
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        'pyarrow>=16.1.0',
        'datasets>=2.18',
        'torchao==0.16.0',
    ])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *REAL_DEPS])
    real_deps_flag.touch()
    print('Installed real-run dependencies. Colab runtime will restart now; after it reconnects, run the notebook from the top again.')
    os.kill(os.getpid(), 9)
elif INSTALL_DEPS and needs_real_stack:
    print('Real-run dependencies already prepared for this Colab runtime.')

print('Real-stack needed:', needs_real_stack)

## 6. Build Runtime Config

This creates a temporary config from `configs/{RUN_MODE}_colab.yaml` and applies the notebook controls above.

In [ ]:
import yaml
from pathlib import Path

from autodata.config import save_config

config_path = Path('configs') / f'{RUN_MODE}_colab.yaml'
if not config_path.exists():
    raise FileNotFoundError(f'Missing config: {config_path}')

with config_path.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

config['project']['output_dir'] = OUTPUT_DIR
config['models']['use_real_model'] = bool(USE_REAL_MODEL)
config['dataset']['use_mock_data'] = not bool(USE_REAL_MODEL)
config['generation']['use_mock_generation'] = bool(USE_MOCK_GENERATION)
config['generation']['provider'] = 'mock' if USE_MOCK_GENERATION else GENERATION_PROVIDER
config['training']['enabled'] = bool(USE_REAL_TRAINING)
config['training']['dry_run'] = not bool(USE_REAL_TRAINING)

if EVAL_SAMPLES_PER_DOMAIN:
    config['dataset']['eval_samples_per_domain'] = int(EVAL_SAMPLES_PER_DOMAIN)
if TRAIN_POOL_SAMPLES_PER_DOMAIN:
    config['dataset']['train_pool_samples_per_domain'] = int(TRAIN_POOL_SAMPLES_PER_DOMAIN)
if TOTAL_SYNTHETIC_BUDGET:
    config['generation']['total_budget'] = int(TOTAL_SYNTHETIC_BUDGET)
if MIN_SAMPLES_PER_DOMAIN:
    config['planning']['min_samples_per_domain'] = int(MIN_SAMPLES_PER_DOMAIN)
if MAX_TRAINING_STEPS:
    config['training']['max_steps'] = int(MAX_TRAINING_STEPS)

runtime_config_path = Path('/content/autodata_runtime_config.yaml')
save_config(config, runtime_config_path)
print('Runtime config:', runtime_config_path)
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

## 7. Optional Smoke Tests

This verifies the package before running the pipeline. It is quick in smoke mode.

In [ ]:
RUN_TESTS = True

if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## 8. Run AutoData

Single run executes the full loop once. Strategy comparison runs one loop for each listed strategy and writes a comparison report.

In [ ]:
from pathlib import Path

if RUN_STRATEGY_COMPARISON:
    from autodata.experiments.strategy_comparison import run_strategy_comparison

    strategy_list = [item.strip() for item in STRATEGIES.split(',') if item.strip()]
    run_output = run_strategy_comparison(config, strategies=strategy_list)
    print('Comparison directory:', run_output['comparison_dir'])
else:
    from autodata.loop.run_loop import run_autodata_loop

    result = run_autodata_loop(config)
    run_output = {'run_dir': result.run_dir}
    print('Run directory:', result.run_dir)
    print('Base accuracy:', f'{result.evaluation_base.overall_accuracy:.3f}')
    print('After accuracy:', f'{result.evaluation_after.overall_accuracy:.3f}')
    print('Training status:', result.training_report.status)

## 9. Inspect Results

The important files are `round_summary.json`, `data_plan.json`, `verification_report.json`, `mixture_report.json`, and `next_round_recommendation.json`.

In [ ]:
import json
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

if RUN_STRATEGY_COMPARISON:
    comparison_path = Path(run_output['comparison_dir']) / 'strategy_comparison.json'
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    rows = comparison['strategies']
    if pd:
        display(pd.DataFrame(rows)[[
            'strategy',
            'base_overall_accuracy',
            'after_overall_accuracy',
            'overall_gain',
            'weakest_domain_improvement',
            'strong_domain_drop',
            'accepted_ratio',
            'training_status',
        ]])
    else:
        print(json.dumps(rows, indent=2))
else:
    run_dir = Path(run_output['run_dir'])
    summary = json.loads((run_dir / 'round_summary.json').read_text(encoding='utf-8'))
    data_plan = json.loads((run_dir / 'data_plan.json').read_text(encoding='utf-8'))
    verification = json.loads((run_dir / 'verification_report.json').read_text(encoding='utf-8'))
    mixture = json.loads((run_dir / 'mixture_report.json').read_text(encoding='utf-8'))
    next_round = json.loads((run_dir / 'next_round_recommendation.json').read_text(encoding='utf-8'))

    print('Run dir:', run_dir)
    print('Summary:')
    print(json.dumps({
        'base_overall_accuracy': summary['base_overall_accuracy'],
        'after_overall_accuracy': summary['after_overall_accuracy'],
        'overall_gain': summary['overall_gain'],
        'generated_count': summary['generated_count'],
        'accepted_count': summary['accepted_count'],
        'mixture_sample_count': summary['mixture_sample_count'],
        'training_status': summary['training_status'],
        'next_round_focus_domains': summary['next_round_focus_domains'],
    }, indent=2))

    if pd:
        plan_rows = []
        for domain, item in data_plan['plan'].items():
            plan_rows.append({
                'domain': domain,
                'num_samples': item['num_samples'],
                'data_type': item['data_type'],
                'reason': item['reason'],
                'accepted': verification['accepted_by_domain'].get(domain, 0),
                'mixture': mixture['domain_distribution'].get(domain, 0),
                'gain': next_round['per_domain_gain'].get(domain, 0.0),
                'efficiency': next_round['learning_efficiency_by_domain'].get(domain, 0.0),
            })
        display(pd.DataFrame(plan_rows))
    else:
        print('Data plan:', json.dumps(data_plan, indent=2))

## 10. Peek At Generated Samples

Use this to sanity-check the generated SFT format before training.

In [ ]:
if not RUN_STRATEGY_COMPARISON:
    run_dir = Path(run_output['run_dir'])
    sample_path = run_dir / 'verified_samples.jsonl'
    print('Sample file:', sample_path)
    with sample_path.open('r', encoding='utf-8') as handle:
        for idx, line in enumerate(handle):
            print(json.dumps(json.loads(line), indent=2)[:1200])
            if idx >= 2:
                break
else:
    print('Open each run_dir from the comparison table to inspect its verified_samples.jsonl file.')

## 11. Zip The Latest Run

This is optional. It creates a zip archive beside the run directory for download or sharing.

In [ ]:
from shutil import make_archive

if not RUN_STRATEGY_COMPARISON:
    run_dir = Path(run_output['run_dir'])
    archive_path = make_archive(str(run_dir), 'zip', root_dir=run_dir)
    print('Created:', archive_path)
else:
    comparison_dir = Path(run_output['comparison_dir'])
    archive_path = make_archive(str(comparison_dir), 'zip', root_dir=comparison_dir)
    print('Created:', archive_path)